In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from bertviz import head_view, model_view

In [ ]:
if torch.cuda.is_available():
    print("CUDA is available.")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    x = torch.tensor([1.0, 2.0]).to("cuda")
    print(f"Test Tensor Device: {x.device}")
else:
    print("CUDA is NOT available. Check your 'uv' install or drivers.")

In [ ]:
# 1. Configuration for 4-bit loading (Crucial for 12GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "Qwen/Qwen2.5-7B-Instruct"  # "meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
# 2. Load Model
# device_map="auto" will automatically place layers on your GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    use_cache=False,        # Disable KV cache (not needed for viz, saves RAM)
    output_attentions=True  # REQUIRED: Forces model to output attention weights
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# 3. Define Prompt
# Keep it short! Attention matrices grow quadratically (SeqLen^2).
# A 12GB card might choke on visualization if prompt > 512 tokens.
# prompt = """
# this is something else

# ⚠️ **CRITICAL: YOU ARE THE DOCUMENT, NOT THE REVIEWER** ⚠️

# **YOUR OUTPUT BECOMES THE ACTUAL DOCUMENT FILE**
# """
prompt = """
this is something else

<critical> you are the document, not the reviewer </critical>

your output becomes the actual document file
"""

inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

In [ ]:
# 4. Inference
# Standard PyTorch inference
with torch.no_grad():
    outputs = model(**inputs)

# 5. Extract and Visualize
attention = outputs.attentions  # Tuple of (Batch, NumHeads, SeqLen, SeqLen)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

In [ ]:
# Launch interactive interface
head_view(attention, tokens)

In [ ]:
model_view(attention, tokens)